# 🇹🇼 台灣台語語音轉文字轉錄器
## MediaTek Breeze-ASR-26 (台語 Taigi) + 時間戳記對齊

---

### 📌 功能特色
- ✅ **一鍵執行**：無需重啟 kernel，首次即可成功
- 🇹🇼 **台語專用**：MediaTek Breeze-ASR-26 專為台灣台語 (Taiwanese Hokkien) 優化
- 🚀 **GPU 自適應**：自動識別 T4/L4/A100/H100 並優化設定
- ⚡ **智能並行**：模型下載與檔案上傳同時進行，大幅縮短等待時間
- ⏱️ **時間戳記對齊**：每段文字標註起始與結束時間
- 📺 **SRT 字幕輸出**：可選輸出標準 SRT 字幕格式
- 📊 **超長音檔支援**：智能分段處理，支援數小時音檔
- 🗣️ **台語轉漢字**：將台語語音轉錄為漢字（華語漢字）輸出

---


In [ ]:
# @title
# =============================================================================
# 台灣台語語音轉文字轉錄器 v6.1 (Breeze-ASR-26 加速版)
# 使用 MediaTek Breeze-ASR-26 (台語 Taigi) 模型 + 精準句子級時間戳記 + 步幅切片技術
# =============================================================================
# 版本: 6.1.0 (2026-04)
# 加速優化 (基於 v6.0):
# 1. generate_kwargs 加入 language="zh" + task="transcribe"，跳過語言偵測開銷。
# 2. torch.compile() 圖層級融合優化 (PyTorch 2.x)，重複推論模式加速。
# 3. T4 batch_size 從 4 提升至 8 (Greedy Decoding 下 VRAM 足夠)。
# 4. no_repeat_ngram_size=3 防止重複幻覺 (取代 condition_on_prev_tokens=False)。
# 5. chunk_length_s=30, stride_length_s=(4,2) 縮小重疊區減少冗餘。
# 6. 模型載入後進行 CUDA warmup，避免首次推論的 JIT 編譯延遲。
# =============================================================================

import os
import sys
import subprocess
import warnings
import time
import gc
import threading
import logging
from typing import Optional, Tuple, Dict, Any, List
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from IPython.display import display, clear_output

# 抑制過多的 Hugging Face 警告與優化 PyTorch 記憶體分配
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# =============================================================================
# 第一部分：環境設定 (強制版本校對)
# =============================================================================

def install_dependencies():
    """使用 pip 原生機制進行快速且安全的依賴檢查與安裝"""
    print("📦 正在檢查並安裝依賴套件 (確保版本相容)...")
    start_time = time.time()

    # 檢查 ffmpeg
    try:
        subprocess.run(["ffmpeg", "-version"], capture_output=True, check=True)
    except:
        print("   📥 安裝系統套件 ffmpeg...")
        subprocess.run(["apt-get", "-y", "update", "-qq"], capture_output=True)
        subprocess.run(["apt-get", "-y", "install", "ffmpeg", "-qq"], capture_output=True)

    # 強制依賴清單 (交給 pip 判斷，若已滿足條件 pip 只需 1 秒即可跳過)
    requirements = [
        "transformers>=4.36.0",
        "accelerate>=0.21.0",
        "pydub",
        "soundfile",
    ]

    print("   📥 校對 Python 套件版本...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade"] + requirements,
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

    # 清除 Colab 安裝時雜亂的輸出 (保持 UI 乾淨)
    clear_output()
    print(f"✅ 依賴環境準備完成！(耗時 {time.time() - start_time:.1f} 秒)")

install_dependencies()

# =============================================================================
# 第二部分：匯入函式庫
# =============================================================================

import tempfile
import torch
import soundfile as sf
from pydub import AudioSegment
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("✅ 所有核心函式庫載入成功！")

# =============================================================================
# 第三部分：資料結構定義
# =============================================================================

@dataclass
class TimestampedSegment:
    """帶時間戳記的轉錄片段"""
    start_time: float  # 秒
    end_time: float    # 秒
    text: str

    def format_time(self, seconds: float, srt_format: bool = False) -> str:
        """格式化時間"""
        # 防呆機制：確保秒數不是 None 或負數
        if seconds is None or seconds < 0:
            seconds = 0.0

        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = seconds % 60

        if srt_format:
            return f"{hours:02d}:{minutes:02d}:{secs:06.3f}".replace('.', ',')
        else:
            return f"{hours:02d}:{minutes:02d}:{int(secs):02d}"

    def to_timestamp_line(self) -> str:
        start = self.format_time(self.start_time)
        end = self.format_time(self.end_time)
        return f"[{start} - {end}] {self.text}"

    def to_srt_block(self, index: int) -> str:
        start = self.format_time(self.start_time, srt_format=True)
        end = self.format_time(self.end_time, srt_format=True)
        return f"{index}\n{start} --> {end}\n{self.text}\n"

# =============================================================================
# 第四部分：GPU 偵測與安全參數配置
# =============================================================================

class GPUOptimizer:
    @staticmethod
    def detect() -> Tuple[str, torch.dtype, int]:
        if not torch.cuda.is_available():
            print("⚠️ 未偵測到 GPU，使用 CPU 模式 (極慢)")
            return "cpu", torch.float32, 1

        gpu_name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"🔍 偵測到 GPU: {gpu_name} (可用 VRAM: {vram:.1f} GB)")

        # 安全的 Batch Size 啟發式演算法 (考慮到 Whisper 長音檔推論的 VRAM 峰值)
        if vram >= 38:
            batch_size = 24  # A100/H100 (40GB+)
        elif vram >= 22:
            batch_size = 8   # L4 (24GB) - 保守設定以防 OOM
        else:
            batch_size = 8   # T4 (15GB) - Greedy Decoding 下 VRAM 充足

        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

        print(f"✅ 硬體最佳化配置: 精度 {dtype}, 安全批次量 {batch_size}")
        return "cuda:0", dtype, batch_size

# =============================================================================
# 第五部分：音檔轉換
# =============================================================================

class AudioConverter:
    TARGET_SR = 16000

    @staticmethod
    def convert(src_path: str) -> Tuple[str, float]:
        print("🔄 正在對齊音訊格式 (16kHz 採樣率, 單聲道)...")
        start = time.time()

        fd, wav_path = tempfile.mkstemp(suffix=".wav")
        os.close(fd)

        cmd = [
            "ffmpeg", "-y", "-threads", str(os.cpu_count() or 4),
            "-i", src_path, "-ar", str(AudioConverter.TARGET_SR),
            "-ac", "1", "-c:a", "pcm_s16le", "-loglevel", "error", wav_path
        ]

        try:
            subprocess.run(cmd, check=True, capture_output=True)
        except subprocess.CalledProcessError:
            print("   ⚠️ ffmpeg 轉換失敗，自動切換至備用引擎 (pydub)...")
            audio = AudioSegment.from_file(src_path)
            audio = audio.set_frame_rate(AudioConverter.TARGET_SR).set_channels(1).set_sample_width(2)
            audio.export(wav_path, format="wav")

        # 使用 soundfile 獲取時長 (最穩定、無需外部依賴)
        try:
            duration = sf.info(wav_path).duration
        except:
            duration = 0.0

        print(f"✅ 格式轉換完成！音檔總長度: {duration:.1f} 秒 (耗時 {time.time() - start:.1f} 秒)")
        return wav_path, duration

# =============================================================================
# 第六部分：核心引擎 (Breeze-ASR-26 台語) 與非同步監控
# =============================================================================

class BreezeASR:
    MODEL_ID = "MediaTek-Research/Breeze-ASR-26"

    def __init__(self, device: str, torch_dtype: torch.dtype, batch_size: int):
        self.device = device
        self.torch_dtype = torch_dtype
        self.batch_size = batch_size
        self.pipe = None
        self._load_lock = threading.Lock()
        self._is_loaded = False
        self._load_status = "等待初始化"

    def load(self) -> 'BreezeASR':
        """背景安全載入模型"""
        with self._load_lock:
            if self._is_loaded:
                return self

            try:
                self._load_status = "正在下載特徵提取器..."
                processor = AutoProcessor.from_pretrained(self.MODEL_ID)

                self._load_status = "正在載入神經網路模型至 GPU..."
                model = AutoModelForSpeechSeq2Seq.from_pretrained(
                    self.MODEL_ID,
                    torch_dtype=self.torch_dtype,
                    low_cpu_mem_usage=True,
                    attn_implementation="sdpa" if self.device != "cpu" else "eager"
                )
                model.to(self.device)

                # torch.compile 圖層級融合 (PyTorch 2.x 加速)
                self._load_status = "正在套用 torch.compile 最佳化..."
                try:
                    model = torch.compile(model, mode="reduce-overhead")
                    print("   ✅ torch.compile 已啟用")
                except Exception:
                    print("   ⚠️ torch.compile 不支援，使用原生模式")

                self._load_status = "正在構建推論管線 (Pipeline)..."
                self.pipe = pipeline(
                    "automatic-speech-recognition",
                    model=model,
                    tokenizer=processor.tokenizer,
                    feature_extractor=processor.feature_extractor,
                    torch_dtype=self.torch_dtype,
                    device=self.device,
                    chunk_length_s=30,
                    stride_length_s=(4, 2),  # 縮小重疊區 (6,2)→(4,2) 減少冗餘
                    return_timestamps=True
                )

                # CUDA Warmup: 避免首次推論的 JIT/kernel 編譯延遲
                self._load_status = "正在進行 CUDA warmup..."
                try:
                    import numpy as np
                    dummy_audio = np.zeros(16000, dtype=np.float32)  # 1秒靜音
                    _ = self.pipe(dummy_audio, generate_kwargs={
                        "language": "zh", "task": "transcribe", "max_new_tokens": 10
                    })
                    print("   ✅ CUDA warmup 完成")
                except Exception:
                    pass  # warmup 失敗不影響正常使用

                self._is_loaded = True
                self._load_status = "載入完成"
            except Exception as e:
                self._load_status = f"載入失敗: {str(e)}"
                raise e
            return self

    def is_loaded(self) -> bool:
        return self._is_loaded

    def get_status(self) -> str:
        return self._load_status

    def transcribe(self, wav_path: str) -> List[TimestampedSegment]:
        """轉錄音檔 (包含非同步 UX 防假死計時器)"""
        print("\n🎤 啟動 AI 語音辨識引擎 (這可能需要幾分鐘，請耐心等候)...")
        start_time = time.time()

        # 控制標記與線程
        is_running = True

        def display_timer():
            """非同步計時器，防止畫面假死"""
            while is_running:
                elapsed = int(time.time() - start_time)
                mins, secs = divmod(elapsed, 60)
                print(f"\r   ⏳ 引擎全速運算中... [已耗時 {mins:02d}:{secs:02d}]", end="", flush=True)
                time.sleep(1)

        timer_thread = threading.Thread(target=display_timer)
        timer_thread.start()

        segments: List[TimestampedSegment] = []
        try:
            # 執行 Pipeline 轉錄 (加速版)
            # - language="zh" + task="transcribe": 跳過自動語言偵測
            # - no_repeat_ngram_size=3: 防止 n-gram 重複幻覺
            # - 保留 condition_on_prev_tokens=True (預設值)，
            #   避免短音檔重複解碼同一句話
            result = self.pipe(
                wav_path,
                batch_size=self.batch_size,
                generate_kwargs={
                    "max_new_tokens": 440,
                    "language": "zh",
                    "task": "transcribe",
                    "no_repeat_ngram_size": 3,
                }
            )

            # 嚴謹的字典與型別解析防呆
            if "chunks" in result and isinstance(result["chunks"], list):
                for chunk in result["chunks"]:
                    text = chunk.get("text", "").strip()
                    if not text:
                        continue

                    # 安全獲取 timestamp
                    timestamp = chunk.get("timestamp")
                    if isinstance(timestamp, tuple) and len(timestamp) >= 2:
                        start_t = timestamp[0] if timestamp[0] is not None else 0.0
                        end_t = timestamp[1] if timestamp[1] is not None else start_t + 1.0
                    else:
                        # 例外狀況處理
                        start_t, end_t = 0.0, 0.0

                    segments.append(TimestampedSegment(
                        start_time=start_t,
                        end_time=end_t,
                        text=text
                    ))
            else:
                # 萬一模型未回傳 chunks
                safe_text = result.get("text", "無法辨識內容").strip()
                segments.append(TimestampedSegment(0.0, 0.0, safe_text))

        finally:
            # 確保無論成功或報錯，計時器都會停止並清理畫面
            is_running = False
            timer_thread.join()
            print("\r" + " " * 50 + "\r", end="") # 清除計時器字串

        elapsed = time.time() - start_time
        total_chars = sum(len(seg.text) for seg in segments)

        print(f"✅ 轉錄完成！")
        print(f"   總計耗時: {elapsed:.1f} 秒")
        print(f"   產出片段: {len(segments)} 段")
        print(f"   總辨識字數: {total_chars} 字")

        return segments

# =============================================================================
# 第七部分：儲存結果
# =============================================================================

def save_transcript(segments: List[TimestampedSegment], src_name: str, output_srt: bool = True) -> Tuple[Optional[str], Optional[str]]:
    if not segments:
        print("\n⚠️ 轉錄結果為空，不建立檔案")
        return None, None

    base_dir = "/content" if IN_COLAB else os.getcwd()
    base_name = os.path.splitext(os.path.basename(src_name))[0]

    txt_path = os.path.join(base_dir, f"{base_name}_transcript.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        for segment in segments:
            f.write(segment.to_timestamp_line() + "\n")

    srt_path = None
    if output_srt:
        srt_path = os.path.join(base_dir, f"{base_name}_subtitle.srt")
        with open(srt_path, "w", encoding="utf-8") as f:
            for idx, segment in enumerate(segments, 1):
                f.write(segment.to_srt_block(idx) + "\n")

    print(f"\n💾 檔案已儲存：\n - [TXT] {txt_path}" + (f"\n - [SRT] {srt_path}" if srt_path else ""))
    return txt_path, srt_path

# =============================================================================
# 第八部分：主程式
# =============================================================================

def main(output_srt: bool = True):
    print("\n" + "="*65)
    print("🇹🇼 台灣台語語音轉文字轉錄器 v6.1 (加速版)")
    print("   Powered by Breeze-ASR-26 + torch.compile + CUDA warmup")
    print("="*65)

    wav_path = None
    asr = None
    executor = None

    try:
        # 步驟 1: GPU 偵測
        print("\n[步驟 1/5] 硬體資源配置")
        device, dtype, batch_size = GPUOptimizer.detect()
        asr = BreezeASR(device, dtype, batch_size)

        # 步驟 2: 背景載入模型 + 等待上傳
        print("\n[步驟 2/5] 模型就緒與音檔載入")
        executor = ThreadPoolExecutor(max_workers=2)
        model_future = executor.submit(asr.load)

        print("\n⚡ 系統正在背景預熱 AI 台語模型 (Breeze-ASR-26)...")
        print("💡 請現在上傳您的音檔，兩者將並行處理節省您的時間！\n")

        if IN_COLAB:
            uploaded = files.upload()
            if not uploaded:
                print("❌ 操作取消：未選擇任何檔案。")
                return
            src_path = list(uploaded.keys())[0]
        else:
            src_path = input("請輸入音檔本機路徑: ")
            if not os.path.exists(src_path):
                print("❌ 錯誤：找不到該檔案。")
                return

        print(f"\n✅ 成功接收檔案: {src_path}")
        print(f"   目前模型狀態: {asr.get_status()}")

        # 步驟 3: 轉換音檔
        print("\n[步驟 3/5] 音檔前處理")
        wav_path, duration = AudioConverter.convert(src_path)

        # 步驟 4: 等待模型並轉錄
        print("\n[步驟 4/5] 語音特徵辨識與時間戳對齊")
        if not asr.is_loaded():
            print("⏳ 正在等待神經網路載入完成...")
            while not asr.is_loaded():
                print(f"\r   狀態: {asr.get_status()}        ", end="", flush=True)
                time.sleep(1)
            print()

        model_future.result() # 確保背景載入無拋出例外
        segments = asr.transcribe(wav_path)

        # 步驟 5: 儲存與下載
        print("\n[步驟 5/5] 封裝與輸出")
        txt_path, srt_path = save_transcript(segments, src_path, output_srt)

        if txt_path and segments:
            print("\n" + "="*65)
            print("📝 轉錄結果預覽 (擷取前 8 段)")
            print("="*65)
            for seg in segments[:8]:
                print(seg.to_timestamp_line())
            if len(segments) > 8:
                print(f"\n... (內容過長已截斷，共 {len(segments)} 段，請參閱下載檔案) ...")
            print("="*65)

            if IN_COLAB:
                print("\n📥 正在觸發瀏覽器下載...")
                files.download(txt_path)
                if srt_path:
                    files.download(srt_path)

        print("\n✨ 任務圓滿完成，感謝您的使用！")

    except Exception as e:
        print(f"\n❌ 發生未預期的系統錯誤: {e}")
        import traceback
        traceback.print_exc()

    finally:
        # 嚴格的資源與記憶體清理 (避免影響 Colab 下一次執行)
        if wav_path and os.path.exists(wav_path):
            try:
                os.remove(wav_path)
                print("\n🧹 已清理磁碟上的暫存音檔")
            except:
                pass

        if executor:
            executor.shutdown(wait=False)

        if asr and asr.pipe:
            del asr.pipe

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

if __name__ == "__main__":
    main(output_srt=True)

✅ 依賴環境準備完成！(耗時 4.8 秒)
✅ 所有核心函式庫載入成功！

🇹🇼 台灣台語語音轉文字轉錄器 (Breeze-ASR-26 台語版)
   Powered by MediaTek Breeze-ASR-26 (Taigi) & HF Pipeline

[步驟 1/5] 硬體資源配置
🔍 偵測到 GPU: Tesla T4 (可用 VRAM: 14.6 GB)
✅ 硬體最佳化配置: 精度 torch.bfloat16, 安全批次量 4

[步驟 2/5] 模型就緒與音檔載入

⚡ 系統正在背景預熱 AI 台語模型 (Breeze-ASR-26)...
💡 請現在上傳您的音檔，兩者將並行處理節省您的時間！



`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Saving 4月16日 上午6-26.m4a to 4月16日 上午6-26 (1).m4a

✅ 成功接收檔案: 4月16日 上午6-26 (1).m4a
   目前模型狀態: 正在載入神經網路模型至 GPU...

[步驟 3/5] 音檔前處理
🔄 正在對齊音訊格式 (16kHz 採樣率, 單聲道)...
✅ 格式轉換完成！音檔總長度: 5.7 秒 (耗時 0.7 秒)

[步驟 4/5] 語音特徵辨識與時間戳對齊
⏳ 正在等待神經網路載入完成...
   狀態: 正在載入神經網路模型至 GPU...        

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✨ 任務圓滿完成，感謝您的使用！

🧹 已清理磁碟上的暫存音檔
